# T39 - 成交数据
## 第五章：结构分析

本章介绍债券成交数据的结构分析，包括：n1. 交易方式分析
2. 买卖方向分析
3. 行业分布分析
4. 评级分布分析
5. 期限结构分析

## 1. 导入必要的库

In [ ]:
# 数据处理
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging
from typing import Dict, List, Optional

# 数据库连接
import sqlalchemy

# 导入配置
from config import config

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('结构分析')

print('导入成功！')

## 2. 加载数据

In [ ]:
def load_trade_data(days: int = 60) -> pd.DataFrame:
    """加载成交数据"""
    engine = sqlalchemy.create_engine(
        config.database.get_mysql_yq_connection_string(),
        poolclass=sqlalchemy.pool.NullPool
    )
    
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    query = f'''
        SELECT * FROM yq.cjqb 
        WHERE tradedDate >= '{start_date}' AND tradedDate <= '{end_date}'
    '''
    
    df = pd.read_sql(query, engine)
    
    # 数据预处理
    if not df.empty:
        df['tradedDate'] = pd.to_datetime(df['tradedDate'])
        df['dt'] = df['tradedDate'].dt.strftime('%Y-%m-%d')
        for col in ['tradedAmount', 'tradedPrice', 'tradedYield']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print(f'加载 {len(df)} 条记录')
    return df

# 加载数据
df = load_trade_data(days=60)
print(f'日期范围: {df["dt"].min()} ~ {df["dt"].max()}')
print(f'\n数据列: {list(df.columns)}')

## 3. 结构分析器

In [ ]:
class StructureAnalyzer:
    """成交数据结构分析器"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
    
    def analyze_by_field(self, field: str, agg_fields: List[str] = None) -> pd.DataFrame:
        """
        按指定字段进行聚合分析
        
        Args:
            field: 分组字段
            agg_fields: 聚合字段列表
            
        Returns:
            pd.DataFrame: 聚合结果
        """
        if self.df.empty or field not in self.df.columns:
            return pd.DataFrame()
        
        if agg_fields is None:
            agg_fields = ['tradedAmount']
        
        # 构建聚合字典
        agg_dict = {}
        for f in agg_fields:
            if f in self.df.columns:
                agg_dict[f] = ['sum', 'mean', 'count']
        
        agg_dict['bondCode'] = 'nunique'
        
        # 执行聚合
        result = self.df.groupby(field).agg(agg_dict)
        result.columns = ['_'.join(col).strip() for col in result.columns.values]
        result = result.reset_index()
        
        # 重命名列
        result = result.rename(columns={'bondCode_nunique': 'bond_count'})
        
        # 计算占比
        if 'tradedAmount_sum' in result.columns:
            total = result['tradedAmount_sum'].sum()
            if total > 0:
                result['amount_pct'] = result['tradedAmount_sum'] / total * 100
        
        return result
    
    def analyze_industry_distribution(self) -> pd.DataFrame:
        """
        分析行业分布
        
        Returns:
            pd.DataFrame: 行业分布结果
        """
        industry_field = 'yyIndustry'  # 银行间行业
        
        if industry_field not in self.df.columns:
            print(f'数据中不存在 {industry_field} 字段')
            return pd.DataFrame()
        
        result = self.analyze_by_field(
            industry_field,
            ['tradedAmount', 'tradedYield']
        )
        
        # 按成交金额排序
        result = result.sort_values('tradedAmount_sum', ascending=False)
        
        return result
    
    def analyze_rating_distribution(self) -> pd.DataFrame:
        """
        分析评级分布
        
        Returns:
            pd.DataFrame: 评级分布结果
        """
        rating_field = 'yyRating'  # 银行间评级
        
        if rating_field not in self.df.columns:
            print(f'数据中不存在 {rating_field} 字段')
            return pd.DataFrame()
        
        result = self.analyze_by_field(
            rating_field,
            ['tradedAmount', 'tradedYield']
        )
        
        # 定义评级排序
        rating_order = ['AAA', 'AAA-', 'AA+', 'AA', 'AA-', 'A+', 'A', 'A-', 'BBB+', 'BBB', 'BBB-', 'BB+', 'BB', 'BB-', 'B', 'CCC', 'CC', 'C', 'D']
        
        # 创建排序键
        result['rating_order'] = result[rating_field].apply(
            lambda x: rating_order.index(x) if x in rating_order else 999
        )
        result = result.sort_values('rating_order')
        result = result.drop('rating_order', axis=1)
        
        return result
    
    def analyze_daily_pattern(self) -> pd.DataFrame:
        """
        分析每日交易模式
        
        Returns:
            pd.DataFrame: 每日交易模式
        """
        if self.df.empty:
            return pd.DataFrame()
        
        # 按日期聚合
        daily = self.df.groupby('dt').agg({
            'bondCode': 'nunique',
            'tradedAmount': ['sum', 'mean', 'count']
        })
        
        daily.columns = ['bond_count', 'total_amount', 'avg_amount', 'trade_count']
        daily = daily.reset_index()
        
        # 添加星期信息
        daily['weekday'] = pd.to_datetime(daily['dt']).dt.dayofweek
        daily['weekday_name'] = pd.to_datetime(daily['dt']).dt.day_name()
        
        return daily
    
    def analyze_weekday_pattern(self) -> pd.DataFrame:
        """
        分析周内交易模式
        
        Returns:
            pd.DataFrame: 周内交易模式
        """
        daily = self.analyze_daily_pattern()
        
        if daily.empty:
            return pd.DataFrame()
        
        # 按星期聚合
        weekday_stats = daily.groupby('weekday').agg({
            'total_amount': 'mean',
            'trade_count': 'mean',
            'bond_count': 'mean'
        })
        
        weekday_stats.columns = ['avg_amount', 'avg_trades', 'avg_bonds']
        weekday_stats = weekday_stats.reset_index()
        
        # 添加星期名称
        weekday_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
        weekday_stats['weekday_name'] = weekday_stats['weekday'].apply(lambda x: weekday_names[x])
        
        return weekday_stats

# 创建分析器实例
analyzer = StructureAnalyzer(df)
print('结构分析器初始化完成')

## 4. 行业分布分析

In [ ]:
# 行业分布分析
industry_dist = analyzer.analyze_industry_distribution()

if not industry_dist.empty:
    print('\n=== 行业分布分析 ===')
    print(f'共 {len(industry_dist)} 个行业')
    print('\n成交金额TOP10行业:')
    print(industry_dist.head(10).to_string())

## 5. 评级分布分析

In [ ]:
# 评级分布分析
rating_dist = analyzer.analyze_rating_distribution()

if not rating_dist.empty:
    print('\n=== 评级分布分析 ===')
    print(f'共 {len(rating_dist)} 个评级')
    print('\n评级分布:')
    print(rating_dist.to_string())

## 6. 每日交易模式分析

In [ ]:
# 每日交易模式分析
daily_pattern = analyzer.analyze_daily_pattern()

if not daily_pattern.empty:
    print('\n=== 每日交易模式 ===')
    print(f'共 {len(daily_pattern)} 个交易日')
    print('\n最近10个交易日:')
    print(daily_pattern.tail(10).to_string())

## 7. 周内交易模式分析

In [ ]:
# 周内交易模式分析
weekday_pattern = analyzer.analyze_weekday_pattern()

if not weekday_pattern.empty:
    print('\n=== 周内交易模式 ===')
    print(weekday_pattern.to_string())
    
    # 找出最活跃和最不活跃的交易日
    most_active = weekday_pattern.loc[weekday_pattern['avg_amount'].idxmax()]
    least_active = weekday_pattern.loc[weekday_pattern['avg_amount'].idxmin()]
    
    print(f'\n最活跃: {most_active["weekday_name"]} (平均成交金额: {most_active["avg_amount"]:,.0f})')
    print(f'最不活跃: {least_active["weekday_name"]} (平均成交金额: {least_active["avg_amount"]:,.0f})')

## 8. 成交金额规模分析

In [ ]:
def analyze_amount_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    分析成交金额规模分布
    
    Args:
        df: 成交数据
        
    Returns:
        pd.DataFrame: 成交金额规模分布
    """
    if df.empty or 'tradedAmount' not in df.columns:
        return pd.DataFrame()
    
    # 使用配置中的阈值
    thresholds = config.trade_data.volume_thresholds
    
    # 分类
    bins = [0, thresholds['small'], thresholds['medium'], thresholds['large'], float('inf')]
    labels = ['小额(<100万)', '中额(100万-1000万)', '大额(1000万-1亿)', '超大额(>1亿)']
    
    df = df.copy()
    df['amount_category'] = pd.cut(
        df['tradedAmount'],
        bins=bins,
        labels=labels
    )
    
    # 统计
    result = df.groupby('amount_category').agg({
        'tradedAmount': ['count', 'sum'],
        'bondCode': 'nunique'
    })
    
    result.columns = ['trade_count', 'total_amount', 'bond_count']
    result = result.reset_index()
    
    # 计算占比
    total_count = result['trade_count'].sum()
    total_amount = result['total_amount'].sum()
    
    result['count_pct'] = result['trade_count'] / total_count * 100
    result['amount_pct'] = result['total_amount'] / total_amount * 100
    
    return result

# 成交金额规模分布
amount_dist = analyze_amount_distribution(df)

print('\n=== 成交金额规模分布 ===')
print(amount_dist.to_string())

## 9. 综合结构分析报告

In [ ]:
def generate_structure_report(
    df: pd.DataFrame,
    industry_dist: pd.DataFrame,
    rating_dist: pd.DataFrame,
    weekday_pattern: pd.DataFrame,
    amount_dist: pd.DataFrame
):
    """生成结构分析报告"""
    
    print('\n' + '='*70)
    print('成交数据结构分析报告')
    print('='*70)
    
    # 1. 总体概览
    print('\n1. 总体概览')
    print(f'   总成交笔数: {len(df):,}')
    if 'tradedAmount' in df.columns:
        print(f'   总成交金额: {df["tradedAmount"].sum():,.0f}')
    if 'bondCode' in df.columns:
        print(f'   成交债券数: {df["bondCode"].nunique():,}')
    print(f'   交易日数: {df["dt"].nunique()}')
    
    # 2. 行业分布
    if not industry_dist.empty:
        print('\n2. 行业分布TOP5')
        for i, row in industry_dist.head(5).iterrows():
            industry = row.get('yyIndustry', 'N/A')
            amount = row.get('tradedAmount_sum', 0)
            pct = row.get('amount_pct', 0)
            print(f'   {industry}: {amount:,.0f} ({pct:.1f}%)')
    
    # 3. 评级分布
    if not rating_dist.empty:
        print('\n3. 评级分布')
        for i, row in rating_dist.iterrows():
            rating = row.get('yyRating', 'N/A')
            amount = row.get('tradedAmount_sum', 0)
            pct = row.get('amount_pct', 0)
            print(f'   {rating}: {amount:,.0f} ({pct:.1f}%)')
    
    # 4. 周内模式
    if not weekday_pattern.empty:
        print('\n4. 周内交易模式')
        for i, row in weekday_pattern.iterrows():
            day = row['weekday_name']
            amount = row['avg_amount']
            print(f'   {day}: 平均成交金额 {amount:,.0f}')
    
    # 5. 成交规模分布
    if not amount_dist.empty:
        print('\n5. 成交规模分布')
        for i, row in amount_dist.iterrows():
            category = row['amount_category']
            count_pct = row['count_pct']
            amount_pct = row['amount_pct']
            print(f'   {category}: 笔数占比 {count_pct:.1f}%, 金额占比 {amount_pct:.1f}%')
    
    print('\n' + '='*70)

# 生成报告
generate_structure_report(
    df, industry_dist, rating_dist, weekday_pattern, amount_dist
)